# ATAG Waypoint 2050 (3rd edition, light) - Scenario S1

Lightweight version of the 3rd edition S1 (SAF-deployment scenario). The seven
individual biomass SAF pathways (HEFA/ATJ/FT) are collapsed into a single
**generic biofuel**: its quantity is the sum of the pathway quantities and its
CO2 emission factor is the quantity-weighted mean of the pathway factors. Electrofuel,
fossil kerosene, hydrogen and electric are unchanged, so total emissions match the
full 3rd edition S1.

The generic biofuel energy file (`data_inputs/s1_energy.yaml`) is produced with
`aeromaps.utils.energy_aggregation.aggregate_carriers_to_generic`.

## (Re)generate the generic-biofuel energy file

Run this cell to rebuild `data_inputs/s1_energy.yaml` from the full 3rd edition
pathways. It is idempotent and can be skipped if the file already exists.

In [ ]:
# The scenario ships with the package; copy it somewhere writable before running,
# so this notebook's outputs and regenerated inputs land in ./workdir rather than
# in the installed AeroMAPS.
from aeromaps.utils.scenarios import prepare_scenario
from aeromaps.utils.energy_aggregation import aggregate_carriers_to_generic

SCENARIO = prepare_scenario("atag_3rd_edition_light")

BIOMASS_PATHWAYS = [
    "hefa_oil_crops_trees",
    "atj_cellulosic_cover_crops",
    "hefa_waste_residue_lipids",
    "atj_agricultural_residues",
    "ft_woody_biomass",
    "ft_municipal_solid_waste",
    "atj_waste_gas",
]

aggregate_carriers_to_generic(
    energy_carriers_file="../3rd_edition_full/data_inputs/s1_energy.yaml",
    carriers_to_merge=BIOMASS_PATHWAYS,
    generic_name="generic_biofuel",
    output_file=str(SCENARIO / "data_inputs" / "s1_energy.yaml"),
    resource_name="generic_biomass",
)
print("Generated data_inputs/s1_energy.yaml")

## Load and compute

In [ ]:
%matplotlib widget
from aeromaps import create_process

process = create_process(configuration_file=str(SCENARIO / "config_files" / "config_s1.yaml"))
process.compute()
process.write_json()

## Results

In [ ]:
process.plot("air_transport_co2_emissions")

In [ ]:
process.plot("fuel_shares")

In [ ]:
process.plot("energy_expenses")

### Where the fuel price comes from

Per-pathway minimum fuel selling price, split into the pathway's own cost and each
resource it consumes. Note that green electricity and DAC-CO2 carry a zero price here:
the report-derived pathway costs already include them, so pricing the resources again
would double-count. Green electricity likewise carries a zero emission factor, since
the report's carbon-intensity table is already a life-cycle figure. See the note in
`data_inputs/resources.yaml`.

In [ ]:
process.plot("mfsp_detailled")